
### Specialized Libraries Used

- **torch.nn**  
    Defines neural network layers, loss functions, and model architectures.

- **torch.utils.data (Dataset, DataLoader)**  
    Creates custom datasets and loads data in mini-batches for efficient training.

- **transformers (RobertaTokenizerFast, RobertaForSequenceClassification, get_scheduler)**  
    Loads pre-trained RoBERTa models, tokenizes text, and manages learning rate scheduling for NLP tasks.

- **torch.optim (AdamW)**  
    Optimizer designed for training Transformer-based models.

- **joblib**  
    Efficiently saves and loads Python objects such as trained models or encoders.


In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification, get_scheduler
from torch.optim import AdamW
import matplotlib.pyplot as plt
import joblib

2025-09-26 19:26:43.380888: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758914803.578078      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758914803.631973      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
file_path = "/kaggle/input/final-resume-screener-dataset/final_removed cloumns.csv"
df = pd.read_csv(file_path)



print("Dataset shape:", df.shape)
print("Column names:", df.columns.tolist())
print("\nFirst record:")
print(df.head(1).to_string(index=False))


Dataset shape: (12365, 8)
Column names: ['Category', 'Education', 'Experience', 'Certifications', 'Achievements', 'Languages', 'Interests', 'completeness_score']

First record:
  Category                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# -----------------------------
# Multi-GPU Setup
# -----------------------------
def setup_device():
    if torch.cuda.is_available():
        device_count = torch.cuda.device_count()
        print(f"CUDA is available. Found {device_count} GPU(s):")
        for i in range(device_count):
            print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        
        device = torch.device("cuda:0")
        use_multi_gpu = device_count > 1
        return device, use_multi_gpu, device_count
    else:
        print("CUDA not available, using CPU")
        return torch.device("cpu"), False, 0

device, use_multi_gpu, gpu_count = setup_device()


CUDA is available. Found 1 GPU(s):
  GPU 0: Tesla P100-PCIE-16GB


# 📄 Resume Data Preparation & Stratified Splitting

## 🔹 Step 1: Combine Resume Text
- Merge multiple fields (Education, Experience, Certifications, etc.) into **one string**.
- Add **profile completeness** (Low / Medium / High) as extra context.

---

## 🔹 Step 2: Completeness Binning
- Convert `completeness_score` (0.0 – 1.0) into categories:
  - **Low** → 0.0 – 0.5  
  - **Medium** → 0.5 – 0.7  
  - **High** → 0.7 – 1.0  

---

## 🔹 Step 3: Encode Job Categories
- Use `LabelEncoder` to convert job categories into **numeric labels** for classification.  
  Example:  

---

## 🔹 Step 4: Create Stratification Variable
- Combine **Category + CompletenessBin** into one variable.  
Example:
- This ensures both **job type** and **resume completeness** are balanced across splits.

---

## 🔹 Step 5: Stratified Splitting
- Data is split into:
- **Train (80%)**
- **Validation (10%)**
- **Test (10%)**
- Splits are stratified on `Category_CompletenessBin`.

---

## 🔹 Step 6: Outputs
- Distribution of completeness levels.  
- Categories & label encoding.  
- Sizes of train / validation / test splits.  

---



In [4]:
def combine_resume_text_with_completeness(row):
    """Combine multiple resume fields into a single text with completeness information"""
    text_parts = []
    
    # Add completeness level as a feature for the model to learn from
    completeness_score = row['completeness_score']
    if completeness_score >= 0.7:
        completeness_level = "high"
    elif completeness_score >= 0.5:
        completeness_level = "medium"
    else:
        completeness_level = "low"
    
    text_parts.append(f"Profile completeness: {completeness_level}")
    
    # Add other fields as before
    if pd.notna(row['Education']) and row['Education'].strip():
        text_parts.append(f"Education: {row['Education']}")
    
    if pd.notna(row['Experience']) and row['Experience'].strip():
        text_parts.append(f"Experience: {row['Experience']}")
    
    if pd.notna(row['Certifications']) and row['Certifications'].strip():
        text_parts.append(f"Certifications: {row['Certifications']}")
    
    if pd.notna(row['Achievements']) and row['Achievements'].strip():
        text_parts.append(f"Achievements: {row['Achievements']}")
    
    if pd.notna(row['Languages']) and row['Languages'].strip():
        text_parts.append(f"Languages: {row['Languages']}")
    
    if pd.notna(row['Interests']) and row['Interests'].strip():
        text_parts.append(f"Interests: {row['Interests']}")
    
    return " ".join(text_parts)

df['combined_text'] = df.apply(combine_resume_text_with_completeness, axis=1)

# Create completeness bins for stratified sampling
df['completeness_bin'] = pd.cut(df['completeness_score'], 
                               bins=[0, 0.5, 0.7, 1.0], 
                               labels=['Low', 'Medium', 'High'],
                               include_lowest=True)

print(f"\nCompleteness Distribution:")
print(df['completeness_bin'].value_counts())

# Label encoding
label_encoder = LabelEncoder()
df['encoded_labels'] = label_encoder.fit_transform(df['Category'])

unique_categories = df['Category'].unique()
num_labels = len(unique_categories)
print(f"\nCategories: {unique_categories}")
print(f"Number of labels: {num_labels}")

# Create combined stratification variable (Category + Completeness)
df['strat_variable'] = df['Category'].astype(str) + '_' + df['completeness_bin'].astype(str)

print(f"\nStratification Variable Distribution:")
print(df['strat_variable'].value_counts())

# Stratified split considering both category and completeness
train_texts, val_texts, train_labels, val_labels, train_completeness, val_completeness = train_test_split(
    df["combined_text"].tolist(),
    df["encoded_labels"].tolist(),
    df["completeness_score"].tolist(),
    test_size=0.2,
    random_state=SEED,
    stratify=df["strat_variable"].tolist()
)

val_texts, test_texts, val_labels, test_labels, val_completeness, test_completeness = train_test_split(
    val_texts,
    val_labels,
    val_completeness,
    test_size=0.5,
    random_state=SEED,
    stratify=[df.loc[df['combined_text'].isin(val_texts), 'strat_variable'].iloc[i] for i in range(len(val_texts))]
)

print(f"\nDataset splits:")
print(f"Train: {len(train_texts)} samples")
print(f"Validation: {len(val_texts)} samples") 
print(f"Test: {len(test_texts)} samples")



Completeness Distribution:
completeness_bin
Medium    8297
Low       2191
High      1877
Name: count, dtype: int64

Categories: ['Accountant' 'Advocate' 'Agriculture' 'Apparel' 'Architecture' 'Arts'
 'Automobile' 'Aviation' 'Banking' 'Blockchain' 'BPO'
 'Building and Construction' 'Business Analyst' 'Civil Engineer'
 'Consultant' 'Data Science' 'Database' 'Designing' 'DevOps'
 'Digital Media' 'DotNet Developer' 'Education' 'Electrical Engineering'
 'ETL Developer' 'Finance' 'Food and Beverages' 'Health and Fitness'
 'Human Resources' 'Information Technology' 'Java Developer' 'Management'
 'Mechanical Engineer' 'Network Security Engineer' 'Operations Manager'
 'PMO' 'Public Relations' 'Python Developer' 'React Developer' 'Sales'
 'SAP Developer' 'SQL Developer' 'Testing' 'Web Designing']
Number of labels: 43

Stratification Variable Distribution:
strat_variable
Education_Medium           280
Apparel_Medium             258
Consultant_Medium          254
Public Relations_Medium    244
Sa

### Why max_len = 512?
- **Model limit**: Most transformer models (BERT, RoBERTa) can only handle up to **512 tokens**.  
- **Uniform input**: Padding/truncating to 512 keeps all inputs the same length → easier batching.  
- **Practical choice**: 512 tokens usually cover enough context (e.g., key info in resumes).  
- **If text is longer**: Extra tokens are cut off. For very long documents, consider models like **Longformer/BigBird** or chunking text into parts.


In [5]:

class ResumeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }


### Why RoBERTa-base is Best for Resume Screening

Resumes are **long, unstructured, and filled with domain-specific terms**.  
RoBERTa-base handles them better than other models because:

1. **Handles long & complex text** → Can process resumes with multiple sections (Education, Experience, Skills).  
2. **Understands domain jargon** → Trained on massive data, so it learns terms like "CMA", "ERP", "SQL".  
3. **Balanced size** → 125M parameters = strong accuracy without Kaggle GPU memory issues.  
4. **Excellent for classification** → Captures subtle differences between categories like Accountant vs Developer.  
5. **Robust to messy formats** → Works well even when resumes mix sentences, bullet points, and abbreviations.  

👉 In short: RoBERTa-base gives the **best tradeoff between power, efficiency, and accuracy** for resume classification.


### Why AdamW for Resume Data

Resumes contain **long, varied, domain-specific text**.  

AdamW helps by:  

- **Stabilizing learning** → avoids sudden jumps in parameter updates.  
- **Preventing overfitting** → resumes often have repetitive structures (education, skills), so weight decay helps the model generalize.  
- **Working well with transformers** → Hugging Face recommends AdamW as the standard optimizer for BERT/RoBERTa.  


### 🔎 Steps in Resume Classification Pipeline

1. **Model & Tokenizer Setup**
   - Load `roberta-base` and its tokenizer (`RobertaTokenizerFast`).
   - Model chosen for strong language understanding and efficiency.

2. **Batch Size Scaling**
   - Multi-GPU: Scales batch size as `16 × GPU count`.
   - Single GPU/CPU: Batch size = `8`.

3. **Dataset Preparation**
   - Create train, validation, and test datasets.
   - Limit input length to **512 tokens** (RoBERTa max).

4. **Data Loading**
   - Use PyTorch `DataLoader` for batching & shuffling.
   - Parallel loading with `num_workers=2`.

5. **Model Initialization**
   - Load `RobertaForSequenceClassification` with correct number of labels.
   - Move model to GPU(s).
   - Wrap in `nn.DataParallel` if multiple GPUs are used.

6. **Optimizer & Scheduler**
   - Optimizer: `AdamW` with learning rate `2e-5`.

7. **Training (`train_one_epoch`)**
   - Print loss every 100 batches.

8. **Evaluation**
9. **Performance Analysis by Resume Completeness**



In [6]:
# -----------------------------
# Model Setup with Multi-GPU Support
# -----------------------------
MODEL_NAME = "roberta-base"
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

# Adjust batch size for multi-GPU
if use_multi_gpu:
    BATCH_SIZE = 16 * gpu_count  # Scale batch size with number of GPUs
    print(f"Using {gpu_count} GPUs with total batch size: {BATCH_SIZE}")
else:
    BATCH_SIZE = 8
    print(f"Using single GPU/CPU with batch size: {BATCH_SIZE}")

# Create datasets
train_dataset = ResumeDataset(train_texts, train_labels, tokenizer, max_len=512)
val_dataset = ResumeDataset(val_texts, val_labels, tokenizer, max_len=512)
test_dataset = ResumeDataset(test_texts, test_labels, tokenizer, max_len=512)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Initialize model
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=num_labels
)

# Move model to device and wrap with DataParallel if multiple GPUs
model.to(device)
if use_multi_gpu:
    print(f"Wrapping model with DataParallel for {gpu_count} GPUs")
    model = nn.DataParallel(model)

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=2e-5)

EPOCHS = 5
num_training_steps = len(train_loader) * EPOCHS
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps,
)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

Using single GPU/CPU with batch size: 8


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:


def train_one_epoch(model, loader, optimizer, device, scheduler):
    model.train()
    running_loss = 0.0
    
    for batch_idx, batch in enumerate(loader):
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        # If using DataParallel, loss is averaged across GPUs automatically
        if use_multi_gpu:
            loss = loss.mean()
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        running_loss += loss.item()
        
        # Print progress every 100 batches
        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx}/{len(loader)}, Loss: {loss.item():.4f}")
    
    return running_loss / max(1, len(loader))

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    preds, labs = [], []
    
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        pred = torch.argmax(logits, dim=-1)
        
        preds.extend(pred.cpu().numpy().tolist())
        labs.extend(labels.cpu().numpy().tolist())
    
    return np.array(preds), np.array(labs)

def analyze_performance_by_completeness(predictions, true_labels, completeness_scores, label_encoder):
    """Analyze model performance by completeness levels"""
    print("\n=== PERFORMANCE BY COMPLETENESS LEVEL ===")
    
    # Create completeness bins
    completeness_bins = pd.cut(completeness_scores, 
                              bins=[0, 0.5, 0.7, 1.0], 
                              labels=['Low (<0.5)', 'Medium (0.5-0.7)', 'High (≥0.7)'],
                              include_lowest=True)
    
    for level in ['Low (<0.5)', 'Medium (0.5-0.7)', 'High (≥0.7)']:
        mask = completeness_bins == level
        if mask.sum() > 0:
            level_preds = predictions[mask]
            level_labels = true_labels[mask]
            
            acc = accuracy_score(level_labels, level_preds)
            f1 = f1_score(level_labels, level_preds, average='weighted')
            
            print(f"{level}: {mask.sum()} samples | Accuracy: {acc:.4f} | F1: {f1:.4f}")

In [8]:
# -----------------------------
# Training Loop
# -----------------------------
best_val_acc = 0.0
best_dir = "roberta_resume_screener_best"

print("\n" + "="*50)
print("STARTING TRAINING...")
print("="*50)

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 30)
    
    train_loss = train_one_epoch(model, train_loader, optimizer, device, scheduler)
    val_preds, val_labels_np = evaluate(model, val_loader, device)
    val_acc = accuracy_score(val_labels_np, val_preds)
    val_f1 = f1_score(val_labels_np, val_preds, average='weighted')
    
    print(f"Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
    
    # Analyze performance by completeness level
    analyze_performance_by_completeness(val_preds, val_labels_np, val_completeness, label_encoder)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # Save the underlying model (not the DataParallel wrapper)
        model_to_save = model.module if use_multi_gpu else model
        model_to_save.save_pretrained(best_dir)
        tokenizer.save_pretrained(best_dir)
        print(f"New best model saved with validation accuracy: {val_acc:.4f}")



STARTING TRAINING...

Epoch 1/5
------------------------------
  Batch 0/1237, Loss: 3.8192
  Batch 100/1237, Loss: 3.7110
  Batch 200/1237, Loss: 3.6489
  Batch 300/1237, Loss: 2.9377
  Batch 400/1237, Loss: 2.3748
  Batch 500/1237, Loss: 1.8715
  Batch 600/1237, Loss: 0.8480
  Batch 700/1237, Loss: 2.3917
  Batch 800/1237, Loss: 2.1819
  Batch 900/1237, Loss: 1.0991
  Batch 1000/1237, Loss: 1.2036
  Batch 1100/1237, Loss: 1.0134
  Batch 1200/1237, Loss: 1.4269
Train Loss: 2.0336 | Val Acc: 0.7888 | Val F1: 0.7889

=== PERFORMANCE BY COMPLETENESS LEVEL ===
Low (<0.5): 212 samples | Accuracy: 0.7453 | F1: 0.7611
Medium (0.5-0.7): 832 samples | Accuracy: 0.7981 | F1: 0.7944
High (≥0.7): 192 samples | Accuracy: 0.7969 | F1: 0.8066
New best model saved with validation accuracy: 0.7888

Epoch 2/5
------------------------------
  Batch 0/1237, Loss: 0.1654
  Batch 100/1237, Loss: 0.7291
  Batch 200/1237, Loss: 1.4729
  Batch 300/1237, Loss: 0.2770
  Batch 400/1237, Loss: 1.5810
  Batch 500

In [9]:

# -----------------------------
# Final Test Evaluation
# -----------------------------
print("\n" + "="*50)
print("LOADING BEST MODEL FOR FINAL EVALUATION...")
print("="*50)

# Load the best model
best_model = RobertaForSequenceClassification.from_pretrained(best_dir)
best_model.to(device)
if use_multi_gpu:
    best_model = nn.DataParallel(best_model)

test_preds, test_labels_np = evaluate(best_model, test_loader, device)

# Calculate overall metrics
acc = accuracy_score(test_labels_np, test_preds)
prec = precision_score(test_labels_np, test_preds, average='weighted')
rec = recall_score(test_labels_np, test_preds, average='weighted')
f1 = f1_score(test_labels_np, test_preds, average='weighted')
cm = confusion_matrix(test_labels_np, test_preds)

print("\n=== FINAL TEST EVALUATION ===")
print(f"Overall Accuracy : {acc:.4f}")
print(f"Overall Precision: {prec:.4f}")
print(f"Overall Recall   : {rec:.4f}")
print(f"Overall F1-score : {f1:.4f}")

# Detailed performance analysis by completeness
analyze_performance_by_completeness(test_preds, test_labels_np, test_completeness, label_encoder)

print("\nClassification Report:")
print(classification_report(
    test_labels_np, 
    test_preds, 
    target_names=unique_categories
))

# Save all artifacts
joblib.dump(label_encoder, os.path.join(best_dir, 'label_encoder.pkl'))

# Save completeness analysis results
completeness_analysis = {
    'test_completeness': test_completeness,
    'test_predictions': test_preds,
    'test_labels': test_labels_np
}
joblib.dump(completeness_analysis, os.path.join(best_dir, 'completeness_analysis.pkl'))

print(f"\nTraining completed using {gpu_count if use_multi_gpu else 1} GPU(s)")
print(f"Model, tokenizer, label encoder, and analysis saved to: {best_dir}")

# Final recommendation based on performance
print("\n" + "="*50)
print("COMPLETENESS STRATEGY EVALUATION")
print("="*50)
print("Monitor the performance by completeness levels above.")
print("If low-completeness resumes show significantly worse performance,")
print("consider running the model again with filtering (completeness_score >= 0.5)")
print("="*50)



LOADING BEST MODEL FOR FINAL EVALUATION...

=== FINAL TEST EVALUATION ===
Overall Accuracy : 0.8149
Overall Precision: 0.8225
Overall Recall   : 0.8149
Overall F1-score : 0.8138

=== PERFORMANCE BY COMPLETENESS LEVEL ===
Low (<0.5): 224 samples | Accuracy: 0.7500 | F1: 0.7556
Medium (0.5-0.7): 827 samples | Accuracy: 0.8368 | F1: 0.8346
High (≥0.7): 186 samples | Accuracy: 0.7957 | F1: 0.7877

Classification Report:
                           precision    recall  f1-score   support

               Accountant       0.80      1.00      0.89        36
                 Advocate       0.93      0.83      0.88        30
              Agriculture       0.91      0.74      0.82        27
                  Apparel       0.92      0.71      0.80        31
             Architecture       0.88      0.68      0.76        31
                     Arts       0.85      0.83      0.84        35
               Automobile       0.75      0.64      0.69        33
                 Aviation       0.88      